# 06m3 — Generalization: GTEA (action-segmentation family, Phase 2)

**Role: pipeline validation + representation quality.** GT action labels (11-symbol alphabet:
`take, open, pour, ...` + `background`->0) used directly as symbols, RLE'd into segment sequences.
The barycenter of an activity's executions is a **prototypical execution**.

GTEA has 28 videos across **7 activities (~4 each)** — below the order-null's 5-fold minimum, so its
order table is **empty by construction** (honest: underpowered). GTEA therefore validates the loader,
ground cost, and the quality half end-to-end; **Breakfast (06m)** carries the order-null counterpoint
to SDS2's 0/15. Executed cheapest-first (this notebook, then 06m2, then 06m). Kernel: `smartflat_repro`.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault('NUMBA_THREADING_LAYER', 'workqueue')  # fork-safe rTWE under nbconvert
os.environ.setdefault('MPLBACKEND', 'agg')                   # headless figures
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils import upsample_sequence
from smartflat.features.symbolic_barycenter.generalization.action_segmentation import (
    download_action_seg, load_action_seg, build_action_seg_ground_cost,
    default_root, read_mapping, DATASETS)
from smartflat.features.symbolic_barycenter.generalization.suite import run_generalization_suite
from smartflat.features.symbolic_barycenter.registries import default_baseline_methods
from smartflat.features.symbolic_barycenter.visualization import plot_cohort_barycenters

NAME, UPSAMPLE = 'gtea', 64   # UPSAMPLE ~= 2x median segment count (median 34); realistic barycenter n_segments
OUT = os.path.join(get_data_root(), 'outputs', 'symbolic_barycenter', 'generalization', NAME)
os.makedirs(OUT, exist_ok=True)
print('dataset:', NAME, '| upsample_to:', UPSAMPLE, '| output dir:', OUT)

dataset: gtea | upsample_to: 64 | output dir: /home/perochon/data-gold-final/outputs/symbolic_barycenter/generalization/gtea


## 1. Load + config-vs-real-files sanity

In [2]:
# GT action labels used DIRECTLY as symbols (RLE'd -> segment sequences, background -> 0).
# download_action_seg pulls only the tiny GT text via HTTP Range (never the 30 GB features);
# it is flag-guarded, so this is a no-op once the data is cached.
download_action_seg(NAME)
meta, X, labels, G = load_action_seg(NAME)
cfg = DATASETS[NAME]
ns = meta['n_segments'].to_numpy()
display(pd.DataFrame({
    'metric': ['N videos (loaded)', '|V| = G (incl. background 0)', '#activity classes',
               'segment length p10/50/90', 'published N', 'published #classes'],
    'value':  [len(X), G, len(set(labels)),
               tuple(np.percentile(ns, [10, 50, 90]).round(1)),
               cfg['n_videos'], cfg['n_classes']],
}))
print('activities:', sorted(set(labels)))
print('per-activity video counts:', dict(Counter(labels)))

,metric,value
0,N videos (loaded),28
1,|V| = G (incl. background 0),11
2,#activity classes,7
3,segment length p10/50/90,"(25.0, 34.0, 41.0)"
4,published N,28
5,published #classes,7


activities: ['Cheese', 'CofHoney', 'Coffee', 'Hotdog', 'Pealate', 'Peanut', 'Tea']
per-activity video counts: {'Cheese': 4, 'CofHoney': 4, 'Coffee': 4, 'Hotdog': 4, 'Pealate': 4, 'Peanut': 4, 'Tea': 4}


## 2. Co-occurrence ground cost

In [3]:
# Data-driven co-occurrence ground cost over the symbol alphabet (symbols that frequently
# abut are closer) -> shared vocab.compute_distance_matrix. Built once; reused by the suite.
D_G = build_action_seg_ground_cost(NAME, X=X, kind='cooccurrence')
assert np.allclose(D_G, D_G.T) and np.allclose(np.diag(D_G), 0.0), 'D_G must be symmetric, zero-diagonal'
if D_G.shape != (G, G):   # loader G=len(mapping) vs ground-cost G=max(observed)+1 (a top id unused)
    print(f'NOTE: ground-cost G={D_G.shape[0]} != mapping G={G}; using {D_G.shape[0]} for shape-consistency')
    G = D_G.shape[0]
print('D_G shape:', D_G.shape, '| symmetric, zero-diagonal OK')

D_G shape: (11, 11) | symmetric, zero-diagonal OK


## 3. Generalization suite (order-null + representation quality)

In [4]:
res = run_generalization_suite(X, labels, G, D_G, name=NAME, upsample_to=UPSAMPLE)
res['order'].to_csv(os.path.join(OUT, 'order_null.csv'), index=False)
res['quality'].reset_index().to_csv(os.path.join(OUT, 'quality.csv'), index=False)
print(f"ORDER-NULL: {len(res['order'])} rows -- EMPTY expected (7 activities x ~4 videos < 5 folds;"
      ' GTEA is underpowered for the order-null, so it validates the pipeline + quality half).')
print('\nQUALITY (methods x representation-fidelity metrics; lower inertia/freq/struct = better):')
display(res['quality'].round(3))

ORDER-NULL: 0 rows -- EMPTY expected (7 activities x ~4 videos < 5 folds; GTEA is underpowered for the order-null, so it validates the pipeline + quality half).

QUALITY (methods x representation-fidelity metrics; lower inertia/freq/struct = better):


metric,inertia_rtwe,inertia_native,freq_fidelity,struct_preservation,entropy_bits,n_distinct,n_segments,stability_inertia_rtwe,stability_histogram,dataset
method,,,,,,,,,,
dba_dtw,7.949,7.428,0.053,0.525,2.458,7.714,33.429,0.125,0.008,gtea
edit_median,7.969,11.179,0.037,0.333,2.502,7.714,32.143,0.000,0.000,gtea
k_medoid,7.855,7.855,0.041,0.364,2.462,7.714,32.000,0.000,0.000,gtea
majority_voting,12.673,0.318,0.143,0.977,2.078,7.429,32.286,0.000,0.000,gtea
soft_dtw,8.034,-37.182,0.055,0.544,2.447,7.714,33.429,0.188,0.008,gtea
wasserstein,NaN,0.043,0.020,NaN,2.486,11.000,NaN,NaN,0.000,gtea


## 4. Prototypical execution per activity (chronograms)

In [5]:
# Prototypical execution per activity: a deterministic edit-median barycenter of each
# activity's executions, rendered as a chronogram strip (reuses plot_cohort_barycenters).
methods = default_baseline_methods(D_G)
code_to_label = {i: n for n, i in
                 read_mapping(os.path.join(default_root(NAME), 'mapping.txt'), cfg['background']).items()}
CAP_CHRONO, rng = 60, np.random.default_rng(0)
proto = {}
for a in sorted(set(labels)):
    ia = np.where(labels == a)[0]
    if len(ia) > CAP_CHRONO:
        ia = rng.choice(ia, CAP_CHRONO, replace=False)
    Xa = np.vstack([upsample_sequence(X[i], UPSAMPLE) for i in ia]).astype(int)
    proto[a] = np.asarray(methods['edit_median']['build'](Xa, 0)).astype(int)
plot_cohort_barycenters(proto, groups=sorted(proto), code_to_label=code_to_label, mask_background=True,
                        title=f'{NAME}: prototypical execution per activity (edit-median barycenter)',
                        savepath=os.path.join(OUT, 'chronograms.png'))
print('saved', os.path.join(OUT, 'chronograms.png'))

saved /home/perochon/data-gold-final/outputs/symbolic_barycenter/generalization/gtea/chronograms.png


**Reading.** The quality table ranks averagers on how faithfully each summarizes an activity's
executions (rTWE inertia = compactness on the common yardstick; freq_fidelity = histogram match;
struct_preservation = bigram-transition match; entropy/n_distinct = mode-collapse lens). The
order-null is empty **by design** here (too few videos/activity) — not evidence against order.
CSVs: `order_null.csv`, `quality.csv`; figure `chronograms.png` under the dataset output dir.